In [0]:
#Transformation we are going to achive in two ways , using DSL approach and using SQL approach

1. Data Munging - (Cleanup) Process of transforming and mapping data from Raw form into Tidy(usable) format with the intent of making it more appropriate and valuable for a variety of downstream purposes such for further Transformation/Enrichment, Egress/Outbound, analytics, Datascience/AI application & Reporting

Passive Data Munging - Data Discovery/Data Exploration/ EDA (Exploratory Data Analytics) (every layers ingestion/transformation/analytics/consumption) - Performing an (Data Exploration) exploratory data analysis of the raw data to identify the attributes and patterns.


a. passive Data Munging
EDA / DATA Exploration
manually understand the Data - manual EDA
1.header
2.delimiter
3.footer
4.columns and datatypes
5.comments
6.record count
7.duplicates / nulls / format issues

Active Data Munging

Combining Data + Schema Evolution/Merging/Merging (Structuring)
Validation, Cleansing, Scrubbing - Cleansing (removal of unwanted datasets), Scrubbing (convert raw to tidy)
De Duplication and Levels of Standardization () of Data to make it in a usable format (Dataengineers/consumers)

In [0]:
#programatically perform EDA on the Source Data
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",inferSchema=True).toDF("custid","fname","lname","age","profession")
cust_df.show(10)
cust_df.printSchema()

In [0]:
# column names 

print(cust_df.columns)
print(cust_df.dtypes) # datatyps -> (col,datatype)

print(cust_df.schema)  # get the schema , structure in spark format 

In [0]:
# number records in datafrme 

print(cust_df.count())

In [0]:
# check whether DF contains duplicate 

# using distinct - unique records (row level )
# if the entire record is duplicate , then we can remove with distinct 
# distinct() - > record level dedulpication 
# dropDuplicates() without any argument - > records level deduplication

# dropDuplicates([custid]) with argument - > column level deduplication

print(cust_df.distinct().count())  # de duplication on record level 

print(cust_df.dropDuplicates().count())  # de duplication on record level 

# key column custid 
# as per our business logic - custid is unique 
# custid is unique , unique count of custid in a daframe 

df2=cust_df.select("custid")

df2.show(2)

df2.printSchema()

# unique custid count 
print(cust_df.select("custid").distinct().count()) # find column level unique count 
# case 2 , need all columns with uniqness based on custid 
# remove duplicates based on the custid

print(cust_df.dropDuplicates(["custid"]).count()) # deduplicated on column level and return the entire datframe 

cust_df.describe().show()
#or
display(cust_df.describe())
#or
display(cust_df.summary())

In [0]:
cust_df.filter("custid is null").show()

In [0]:
cust_df.select("custid").distinct().show(5)

cust_df.dropDuplicates(["custid"]).show(5)

In [0]:
# select -> return the new datframe with selected columns
df3=cust_df.select("custid","fname")
df3.show()

scnerios to create Dataframe
suppose i have customer data in diffrent files in same dir - read dir

suppose i have customer data in diffrent files in same dir with sub dir as well - read main dir with recursive_lookup enable

suppose i have customer data and sales data in diffrent files in same dir , i want to read only sales data - read dir with file pattern (/data/sales*)

suppose i have customer data and sales data in diffrent files in same dir and sub dir , i want to read only sales data - read main dir with recursivefilelookup and pathGlobfilter="sales*"

suppose i have sales data in diff directories - list of path or list of files

Schema evolution
changes in the sceham
Day 1 to Day 5 files have cid ,cname ,age

Day 5 to day 10 file have cid ,cname ,age , profession

Day 11 - cid , cname , mobile , profession
we achived this writing into some columnar 
parquet / orc file format
while reading the entire data we will use with mergeschema option

In [0]:
#combining Data -> schema Evolution / Structuring
#From day 1 writting data into parq format - id,name
#From day 2 writting data into parq format - id,name,age
# when we read multiple file with parq and orc using merge schema,it will consolidate and provide single dataframe with all column
stud_df=spark.read.parquet("/Volumes/izwd37dev/wd37db/rawdatta/schema_out/",mergeSchema=True)

stud_df.show()

stud_df.printSchema()

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part1.csv

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part2.csv

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part3.csv

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part4.csv

In [0]:
#1. combine data from diffrent files with changes schema fo csv
stud_df= spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part",header=True,inferSchema=True)

stud_df.show()

stud_df.printSchema()

In [0]:
stud_df1=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part4.csv",header=True,inferSchema=True)

stud_df2=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part3.csv",header=True,inferSchema=True)
stud_df1.show()

stud_df1.printSchema()

stud_df2.show()

stud_df2.printSchema()

In [0]:
complete_stud_df=stud_df1.union(stud_df2)
complete_stud_df.show()

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part1.csv

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part5.csv

In [0]:
stud_df1=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part1.csv",header=True,inferSchema=True)

stud_df5=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part5.csv",header=True,inferSchema=True)


stud_df1.printSchema()

stud_df5.printSchema()
# union will work only with same number of columns and datatypes
#this will give error
stud_df1.union(stud_df5).show()

In [0]:
final_df =stud_df1.unionByName(stud_df5,allowMissingColumns=True)
final_df.show()

In [0]:
# SQL union 

# union --> same number of columns and datatypes based on position 

# in sql union --> retrun unique records 

# in spark DS -  union will allow duplicates 

data1=[(100,"raja",25),(101,"mani",35),(102,"patel",45)]
data2=[(500,"alex",32),(502,"anne",25),(503,"rahul",22),(504,"carry",22)]

data3=[(500,"alex",2000),(502,"anne",2025),(503,"rahul",2022),(504,"carry",2002)]


df1=spark.createDataFrame(data1,['id','name','age']) # 3 records
df2=spark.createDataFrame(data2,['id','name','age']) # 4 records
df3=spark.createDataFrame(data3,['id','name','year']) # 4 records

df1.show()
df2.show()
df3.show()



In [0]:
# combine both df into one 

combined_df=df1.union(df2)

combined_df.show()

print(combined_df.count())

In [0]:
print("id,name age with id , name , year")
combined_df.show()
df3.show()

combined_df1=df1.union(df3)


In [0]:
#since id,name age with id , name , year => here data types are matching. hence no issues
combined_df1.show()

In [0]:
data4=[(100,25,"raja"),(101,35,"mani")]
# 
print("column in diffrent order")
df4=spark.createDataFrame(data4,['id','age','name'])

df4.show()

In [0]:
# combined_df1=df1.union(df4) -> will throw error as column type is different 

In [0]:
# combined_df1=df1.union(df4) -> will throw error as column type is different 


# unionByName
#  default , same number of columns and datatypes based on colun name  
# with allow missing column -> works on differnt column as well 

combined_df1.show()
df4.show()

In [0]:
# unionByName
#  default , same number of columns and datatypes based on colun name  
# with allow missing column -> works on differnt column as well 
combined_df1.printSchema()
df4.printSchema()

df1.show()
df4.show()

combined_df2=df1.unionByName(df4)
combined_df2.show()

In [0]:
# id ,name ,age  ---> id,name,year--> output will be id,name,age,year
# Since we have missing column in both DF we have to use allowMissingColumns=True
df1.show()
df3.show()
combined_df3=df1.unionByName(df3,allowMissingColumns=True)

combined_df3.show()

In [0]:
#2. validation , cleansing , scrubbing
#handle missing value , handling null

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified

In [0]:
# clean up data which ever not matching with schema 

# reject process - 1
schema_str="custid int,fname string,lname string,age int,profession string,error_rec string"
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",schema=schema_str,mode="PERMISSIVE",columnNameOfCorruptRecord="error_rec")
cust_df.show(10)
cust_df.printSchema()

In [0]:
valid_df=cust_df.filter("error_rec is null")
valid_df.count() # 10000 rec

In [0]:
erro_rc_df=cust_df.filter("error_rec is not null")
erro_rc_df.show() # 5 rec

In [0]:
#erro_rc_df.cache() PERSIST TABLE is not supported on serverless compute. SQLSTATE: 0A000
erro_rc_df.select("error_rec").show()

In [0]:
erro_rc_df.write.mode("overwrite").csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/error_table")
# take erro_rec column alone into error_table 
# default spark not working properly this case , workaround cahe() and then perform write operation
# cache() will not work in serverless env
# we can try with compute based databricks environment
# erro_rc_df.select("error_rec").write.mode("overwrite").csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/error_table")

In [0]:
# 2 level cleansing / rejection 

# cleaning up / drop the records 

#  null handling - null record removal 

# null - single column or multiple columns may have null , entire rec may have null 

# single null - remove that recod -> delete rec when col is null 
# mulit col null - remove that recod -> delete rec when col is null and col2 is null 

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified

In [0]:
schema_str="custid int,fname string,lname string,age int,profession string"
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",schema=schema_str,mode="PERMISSIVE")

In [0]:
# handle null -> spark DSL  -> na functions 
# na -> not applicable -> null
cust_df.show()

In [0]:
print(cust_df.count())

In [0]:
# na.drop -> 3 arg -> subset , how , threshold
#  subset -> default is all columns -> option as list of columns 
#  how ->default is  any -> options are  ->   all | any 
#               any -> any one column is the subset is null -> remove that record ->  or
#               all -> all columns are null is the subset  -> remove that record -> and

In [0]:
# if all columns are null remove that record 
all_not_nulll_df=cust_df.na.drop(how="all")
print(all_not_nulll_df.count())


In [0]:
# if any one  columns are null remove that record 
not_nulll_df=cust_df.na.drop(how="any")
not_nulll_df.show()

In [0]:
print(not_nulll_df.count())

In [0]:
# cutid is the key colum it should ot have null if cutid is  null remove that record 
cust_id_not_nulll_df = cust_df.na.drop(subset=["custid"])
cust_id_not_nulll_df.show(5)
print(cust_id_not_nulll_df.filter("custid is null").count())

# cutid and age  is the key colum it should ot have null if cutid and age is  null remove that record 
# custid is null and age is null remove that record
cust_id_age_not_nulll_df=cust_df.na.drop(subset=["custid","age"],how="all")
cust_id_age_not_nulll_df.show(4)


In [0]:
# cutid and age  is the key colum it should ot have null if cutid or age is  null remove that record 
# custid is null or age is null remove that record
cust_id_age_not_nulll_df=cust_df.na.drop(subset=["custid","age"],how="any")

cust_id_age_not_nulll_df.show()  # 

print(cust_id_age_not_nulll_df.count())

In [0]:
cust_df.filter("custid is null").show()

In [0]:
# na.fill -> 3 arg -> subset , value , inplace 
# scrubbing - as we are filling hence its scrubbing
# nvl , coalesce 

# Based on the type of data what we are filling, it is putting 0 in all the places
not_null_df =cust_df.na.fill(0)
not_null_df.show()
# here we have given subset

not_null_df =cust_df.select("age","fname","lname").na.fill(0,subset=["age"]).na.fill("NA",subset=["fname","lname"])
not_null_df.show()

In [0]:
schema_str="custid int,fname string,lname string,age int,profession string"
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",schema=schema_str,mode="PERMISSIVE")

cust_df.show(10)

print(cust_df.count())

In [0]:
# na functions 

# na.drop 
# na.fill
# na.replace 

# drop 

# dropped the record if has null all / subset of columns 

# df.na.drop(how,subset,thres)
# default 

null_dropeed_df=cust_df.na.drop() # all columns , any one column is null, dropped
null_dropeed_df.show()
# subset=* , how=any , thres=none
# 
# na.drop = dropna
null_dropeed_dropna_df=cust_df.dropna() 


null_dropeed_df.show(10)
null_dropeed_dropna_df.show(10)
print(null_dropeed_df.count())

print(null_dropeed_dropna_df.count())


# threshold - int 
# thresh=2 -> atleast 2 columns should not be null  

null_dropeed_df=cust_df.na.drop(thresh=2)

null_dropeed_df.show()

null_dropeed_df.count()

In [0]:
data=[
    (100,"crish",25),
    (101,"bala",None),
    (102,None,None),
    (None,None,None),
    (103,"raja",None),
    (104,None,25),
]


df=spark.createDataFrame(data,["id","name","age"])

df.show()

In [0]:
not_null_df=df.na.drop()
not_null_df.show()

In [0]:
df.na.drop(subset=["id"]).show()

In [0]:
# if either id or age is null - dropped 
df.show()
df.na.drop(subset=["id","age"],how="any").show()

In [0]:
# if id and age both is null - this will be dropped 
df.show()
df.na.drop(subset=["id","age"],how="all").show()

In [0]:
# threshold - int 
# thresh=2 -> atleast 2 columns should not be null. if so dont remove
df.show()
df.na.drop(thresh=2).show()

In [0]:
data=[
    (100,"crish",25),
    (101,"bala",None),
    (102,None,None),
    (None,None,None),
    (103,"raja",None),
    (104,None,25),
]


df=spark.createDataFrame(data,["id","name","age"])

df.show()

df.na.drop().show() # any null will be dropped 
df.na.drop(how="all").show() 

df.show()
df.na.drop(subset=["id","age"]).show() 
# if either id or age is null - dropped 

df.na.drop(thresh=1,subset=["id","age"],how="any").show() 
df.na.drop(thresh=1,subset=["id","age"]).show() 
# if either id or age is null - dropped - atleast on has not null dont dropped 

df.show()
df.na.drop(thresh=2).show()


In [0]:
df.show()
df.na.drop(thresh=2).show()

In [0]:
#scrbbing
data=[
    (100,"crish",25),
    (101,"bala",None),
    (102,None,None),
    (None,None,None),
    (103,"raja",None),
    (104,None,25),
]


df=spark.createDataFrame(data,["id","name","age"])

df.show()
# SQL : fill -> nvl,coalesce, case when 
df.na.fill(0).show() # all int columns null will be replaced with 0

# na.fill= fillna
# df.fillna()
df.na.fill(0,subset=["age"]).fillna("unknwon").show() # all int columns null will be replaced with 0

df.fillna("0").show()


In [0]:
# replace 
# na.replace(original,replacement,subset)
df.show()
df.na.replace("crish","krish",subset=["name"]).show()
cust_df.na.replace("Actor","Flim Actor").show()

cust_df.na.replace({"Actor":"Flim Actor","Pilot":"Air Pilot"}).show()

cust_df.na.replace({"Actor":"Flim Actor","Pilot":"Air Pilot"},subset=["profession"]).show()

In [0]:
data=[
    (100,"crish",25),
    (101,"bala",None),
    (102,None,None),
    (None,"BALA",None),
    (103,"raja",None),
    (104,"Bala",25),
]


df=spark.createDataFrame(data,["id","name","age"])

df.show()

df.na.replace("bala","TEST").show()

In [0]:
#Standardaization -Making the data more standard by adding/removing/reordering columns as per the expected standard, unifying into expected format, converting the type as expected etc.,
cust_df.columns
from pyspark.sql.functions import lit,col,current_date,initcap
# additional audit columns 
# load_date, system ,mod_date, user,applicatioid

# how to add columns with staic value in pyspark 

# SQL -> select 'customer' as system,current_date as load_date , * from staging_cust

# DSL (domain specific language) --> select , withColumn
# withColumn(new_col_name,value ) 
# value -> should be in the column format
# select *,'customerdata' as source from tbl

# syntax : df.withColumn(new_col_name,value)
# return type: Dataframe

# withcolumn -> hardcoded using lit 
#            -> taking another column using col
#            -> using built in sql function which return column - create_date()

# custinfo \
# cust_info_south_20260618.csv
# cust_info_west_20260618.csv

# south -> system  , 20260618 -> data_dt , load_dt-> current_date()


cust_enriched_df = cust_df.withColumn("source",lit("customerdata")) \
                        .withColumn("load_dt",current_date())
cust_enriched_df.show(2)

# select *,profession as new_prof
cust_enriched_df3=cust_enriched_df.withColumn("new_prof",col("profession")).withColumn("jobid",lit(101))
cust_enriched_df3.show(2)

cust_enriched_df3.printSchema()


In [0]:
# profession wise total count 
# group by 
cust_enriched_df3.show(5)
cust_enriched_df3.filter("profession is null").count()
cust_enriched_df3.groupBy("profession").count().show(5,False)

# There is a null in the profession column, so we are filling as PILOT
cust_prof_df=cust_enriched_df3.na.fill("PILOT",subset=["profession"]).groupBy("profession").count()
cust_prof_df.where("profession ='PILOT' or profession ='Pilot'").show()
print(cust_prof_df.where("profession ='PILOT' or profession ='Pilot'").count())

In [0]:
cust_enriched_df3.show(2)
cust_enriched_df4=cust_enriched_df3.na.fill("PILOT",subset=["profession"])


# select custid,fname,lname,age,upper(profession) as profession from tbl
cust_enriched_df5 = cust_enriched_df4.withColumn("New_Prof",initcap(col("profession"))) 
cust_enriched_df5.show()

In [0]:
df6= cust_enriched_df5.filter("profession='Pilot'").groupBy("profession").count()
df6.show()

cust_enriched_df5.groupBy("profession").count().show(5,False)


In [0]:
# read csv  -> Df 
 
# default -> all cols string  ,
# inferschema -> true -> based on your data schema will defined automatically

# schema - structtype -> structfield or ddl string ( recomannded for large data )

# rejection rule -default 3 options (mode) spark provoiding while loading data -> permissive / dropmalformed / failfast 

# permissive - allow everything regardless of schema -> null for wrong data
# dropmalformed - drop the corrupted data 
# failfast - throw error if any data is wrong

# RCA om the bad record / collect the bad record to correct later
# permissive + columnNameOfCorruptedRecord -> add new column to the dataframe (error_rec string)

In [0]:
from pyspark.sql.functions import lit,col,current_date

df=spark.range(50)
# id -> int
print("source Dataframe")
df.show(5)
df.printSchema()
print("df schema output")
df.schema
print("df schema output in print mode")
print(df.schema)
# add columns with dt -> current date (builtin fun)
# add column  with cretedBy -> dbuser (hardcode)
# add column id2 -> id * 2 -> col transformation

# select id , id*2 as id2,current_date() as dt, 'dbuser' as createdBy from range_tbl

user="dbuser"
df2=df.withColumn("id",col("id")*2) \
      .withColumn("dt",current_date()) \
      .withColumn("createdBy",lit(user))
    
      
print("enriched  Dataframe")
df2.show(5)
df2.printSchema()
 
#

In [0]:
data=[(1),(2)]
#if the column is int, by default column name it will be _1
#if the column is string, by default column name it will be _c1
df=spark.createDataFrame(data=data)
df.show()

In [0]:
# using withColumn we added columns 
# using select 

from pyspark.sql.functions import lit,col,current_date

df=spark.range(50)
# id -> int
print("source Dataframe")
df.show(5)
df.printSchema()
print(df.schema)

# add columns with dt -> current date (builtin fun)
# add column  with cretedBy -> dbuser (hardcode)
# add column id2 -> id * 2 -> col transformation

df2=df.select("id",(col("id")*2).alias("id2"),current_date().alias("dt"),lit("dbuser").alias("createdBy"))

print("enriched  Dataframe")
df2.show(5)

In [0]:
from pyspark.sql.functions import concat
schema_str="custid int,fname string,lname string,age int,profession string"
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",schema=schema_str,mode="PERMISSIVE")

cust_df.show(5)
cust_df.select("*").show()
cust_df.select("custid","age").show()
cust_df.select(col("custid"),col("fname")).show()
# concat -> fname and lname -> fullname 
cust_df2=cust_df.withColumn("fullname",concat(col("fname"),lit("|"),col("lname")))
cust_df2.show()

In [0]:
#Standadization -2 - uniformality
# gender -> Male | male | MALE | M -> male
from pyspark.sql.functions import concat,upper,lower,initcap
cust_df.show(5)
standize_df2=cust_df.withColumn("profession",upper(col("profession")))
standize_df2=cust_df.withColumn("profession",initcap(col("profession")))
standize_df2.show()

In [0]:
#Standadization -3 - Type
# cast 
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",mode="PERMISSIVE").toDF("custid","fname","lname","age","profession")
cust_df.show(5)
cust_df.printSchema()

# sql -> select cast(age as int) as age from cust
cust_df2=cust_df.filter("age  not rlike '^[0-9]+$'")
cust_df2.show(5)

cust_df2=cust_df.filter("age  rlike '^[0-9]+$'").withColumn("age",col("age").cast("int"))

cust_df2.show(5)

cust_df2.printSchema()

In [0]:
schema_str="custid int,fname string,lname string,age int,profession string"
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",schema=schema_str,mode="PERMISSIVE").toDF("custid","fname","lname","age","profession")
cust_df.show(5)
cust_df.printSchema()

cust_df2=cust_df.na.fill(0,subset=["age"])

cust_df2.show(5)
cust_df2.printSchema()

In [0]:
#Standadization -4 - Naming
from pyspark.sql.functions import col,concat
cust_df.show(5)

# rename column of dataframe
# withcolumnRenamed
# select with alias 
# withcolumn + drop 

custdf_2 = cust_df.withColumnRenamed("fname","First_Name").withColumnRenamed("lname","Last_Name")
custdf_2.show() 
#Or
custdf_3 = cust_df.select("custid",col("fname").alias("First_Name"),col("lname").alias("Last_Name"))
custdf_3.show() 
 

In [0]:
#Standadization -5 - Naming and reordering
# remove a column 
# select only required column 
# drop
from pyspark.sql.functions import col,concat,lit
cust_df2 = cust_df.withColumn("FullName",concat(col("fname"),lit("|"),col("lname")))
cust_df2.show(5)


# reorder 
cust_final_df=cust_df2.select("custid","fullname","age","profession")

cust_final_df.show(5)

usecase :
we recivied data from multiple sources (regions) we have to load data into our storage

csv with header
file name : empdata_region_datadt.csv
cid,fname,lname,age,prof
output :

cid,fullname,age,prof,load_dt,data_dt,source,created_by

mappings :

cid->cid
fullname -> fname+lname
age->age
prof ->prof
load_dt -> current date
data_dt -> take from the file name
source -> take from the file name
created by -> current user

In [0]:
from pyspark.sql.functions import concat,lit,col,current_date,input_file_name,split 

cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/cust_data/",header=True,inferSchema=True)
cust_df.show(5)

In [0]:
cust_full_name_df=cust_df.withColumn("fullname",concat(col("fname"),lit(" "),col("lname")))
cust_full_name_df.show()

cust_final_full_df =cust_full_name_df.select("cid","age","prof","fullname") 

cust_load_dt_df = cust_final_full_df.withColumn("Load_dt",current_date())
cust_load_dt_df.show()



In [0]:
# removing the csv 
#input_file_name - built in function for spark. in databricks its not accepting
#_metadata.file_path
#_metadata.file_name

#Using Python Way
'''name="custdata_APAC_20260711.csv"
name.split(".")
['custdata_APAC_20260711', 'csv']
name.split(".")[0]
'custdata_APAC_20260711'''

cust_source_datadt_df1=cust_load_dt_df.withColumn("file_name",split(col("_metadata.file_name"),"\\.")[0])
display(cust_source_datadt_df1)  

# created_by 
cust_source_datadt_df3=cust_source_datadt_df1.withColumn("created_by",lit("izuser"))
cust_source_datadt_df3.show()

In [0]:
# final df 
cust_df_final=cust_source_datadt_df3.select(col("cid").alias("custid"),"fullname","age","prof",col("file_name").alias("source"),"Load_dt","created_by")
 


cust_df_final.show(20,False)
cust_df_final.count()

cust_df_final.write.mode("append").saveAsTable("izwd37dev.wd37db.cust_detail")

In [0]:
%sql
select count(*) from izwd37dev.wd37db.cust_detail

In [0]:
from pyspark.sql.functions import col,concat,lit,current_date
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/cust_data",header=True,inferSchema=True)

cust_df.select(col("_metadata")).printSchema()

## Data Enrichment** - Detailing of data
Makes your data rich and detailed <br>
a. Add (withColumn,select,selectExpr), Derive (withColumn,select,selectExpr), Remove/Eliminate (drop,select,selectExpr), Rename (withColumnRenamed,select,selectExpr), Modify/replace (withColumn, select/selectExpr) - (very important spark sql functions) <br>
b. split, merge/Concat <br>
c. Type Casting, reformat & Schema Migration

### select vs selectExpr

In [0]:
# fullname and currentdate -> dt in cust_df
cust_df.select(concat(col("fname"),lit(" "),col("lname")).alias("fullname"),current_date().alias("dt")).show()


In [0]:
# sql -> select concat(fname, ' ', lname) as fullname, current_date() as dt from tbl
cust_df.selectExpr("concat(fname, ' ', lname) as fullname", "current_date() as dt").show()

In [0]:
# withcolumn -> adding a new column 
# withColumns -> adding multiple columns -> withColumns(dictionary)
cust_df.show()
from pyspark.sql.functions import upper,col
cust_df.withColumn("upper_prov",upper(col("prof"))).withColumn("fullname",concat(col("fname"),col("lname"))).show()

In [0]:
#Data Customizations

1. user Defined functions (UDF)
upper , lower , concat ,lit , initcap, col ,current_date ... -> built in functions

how to use your python function inside the select / withcolumn

In [0]:
# 1. import udf -> this will convert python fnction into spark udf
from pyspark.sql.functions import udf
# 2. create your python function using def | lambda ...
def wd37_custom_upper(strValue):
    return strValue.upper()+"$"+strValue.lower()

wd37_custom_upper('sundar')

# 3. convert python function to udf 
#  udf will take function and return type as input , default retrun string type
udf_upper=udf(wd37_custom_upper)

print(wd37_custom_upper("databricks spark"))


In [0]:
# 4 we can use converted udf into  pyspark select / withcolumn 
from pyspark.sql.functions import col
cust_df.select("*").show()
cust_df.select("*",udf_upper(col("prof"))).show()

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

udf_test = udf(lambda a: a*a,IntegerType())

df=spark.range(10)
df.show()

In [0]:
df.select("id",udf_test(col("id"))).show()

In [0]:
df.select("id",udf_test("id")).printSchema()

#derive the flag / indicator from the exsting column / value

In [0]:
from pyspark.sql.functions import udf,col,lit
from pyspark.sql.types import StringType
my_schema="custid int,fname string,lname string,age int,prof string"

cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/custs_header",header=False,schema=my_schema)

cust_df.show(5)


In [0]:
cust_df.printSchema()

In [0]:
# categorice age into multiple category

# if age is lees than 18 -> minor 
# if age is between 18 and 55 - midage
# if age is greater tha 55 - senior 

def get_age_group(age):
    if age is None:
        return None
    if age<18:
        return "minor"
    elif age>=18 and age<=55:
        return "midage"
    else:
        return "senior"
get_age_group(160) 

udf_get_age_cat=udf(get_age_group,StringType())

cust_df2=cust_df.withColumn("age_category",udf_get_age_cat(col("age")))

cust_df2.show()

In [0]:
from pyspark.sql.functions import when
# case when age <18 then 'minor' when age <55 then 'midage' else 'senior' end 

# case when using DSL function 

# when(condition1,value_1).otherwise(value_2)
# when(condition1,value_1).when(condition2,value_2).otherwise(value_default)
cust_age_cat_when_dsl_df=cust_df.withColumn("age_Category",when(col("age")<18,"minor").when((col("age")>=18) & (col("age")<=55),"midage").otherwise("senior"))

cust_age_cat_when_dsl_df.show(2)

# using select
cust_age_cat_select_df=cust_df.select("*",when(col("age")<18,"minor").when(col("age")>18,"senier").alias("age_category"))
cust_age_cat_select_df.show(2)

# selectExpr
cust_age_cat_selectExpr_df=cust_df.selectExpr("*","case when age<18 then 'minor' when age>18 then 'senier' else 'midage' end as age_category")
cust_age_cat_selectExpr_df.show(2)
 
     

In [0]:
# case when age <18 then 'minor' when age <55 then 'midage' else 'senior' end 

# select -> col , dsl function 
# selectExpr -> col, sql expression
# df.select(current_date(),expr("concat('a','b')"))
# expr is the dsl fucntion it will take sql expression as input return a column 

from pyspark.sql.functions import expr

cust_df.withColumn("age_cat",expr("case when age <18 then 'minor' when age <55 then 'midage' else 'senior' end ")).show()

In [0]:
df =cust_df.createOrReplaceTempView("cust")
query="""select custid,concat(fname,lname) as fullname,age,prof,
          case when age <18 then 'minor' else 'senier' end as age_cat
           from cust"""
spark.sql(query).show()

In [0]:
df=spark.range(10)
df.select("id",expr("id*id as new_id"),expr("'test' as name")).show(1)
df.withColumn("id",col("id")*col("id").alias("id")).show(1)
df.withColumn("id",col("id")*col("id").alias("id")).withColumn("name",lit('test')).show()

In [0]:
# 3. data Formating

# date function 
# default date format - yyyy-MM-dd
# convert string  to date - to_date(string,format)
# convert date to string format - date_format(date,format)


In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import to_date,date_format

source_schema= StructType([
    StructField("eid",IntegerType(),True),
    StructField("name",StringType(),True),
    StructField("dob",StringType(),True)
])

emp_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/emp_date.csv",header=True,schema=source_schema)
emp_df.show()
# Spark default date format - yyyy-MM-dd
# what we received is mm-yyyy-dd
emp_df.printSchema()

# convert dob into a date 

# date -> yyyy-MM-dd
emp_df1=emp_df.withColumn("dt_dob",to_date(col("dob"),"MMyyyydd"))
emp_df1.show()

# sending to downstream they want date MM/dd/yyyy

emp_df3=emp_df1.withColumn("str_date",date_format(col("dt_dob"),"MMM/dd/yyyy"))

emp_df3.show()


In [0]:
DATA CORE CURATION (PRE WRANGLING)

Select

Filter

Format

Group and Agg

Sorting

In [0]:
# to_date   - convert string into a date ----> date type
# date_format   - convert date into string -----> string type 
my_schema=StructType([
    StructField("custid",IntegerType(),True),
    StructField("fname",StringType(),True),
    StructField("lname",StringType(),True), 
    StructField("age",IntegerType(),True),
    StructField("profession",StringType(),True),
    StructField("dob",StringType(),True)])

from pyspark.sql.functions import col,when,expr,current_date,date_format
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/custs_header",header=False,schema=my_schema,mode="PERMISSIVE") 

cust_df.show(5)
cust_df.printSchema()

# select only - custid,fname,age

# SQL -> select custid,fname, age from tbl

# col(colname),colname,df.colname

print(cust_df.age)

cust_sel_DF=cust_df.select(col("custid"),col("fname"),cust_df.age,"profession")

cust_sel_DF.show(1)



# is_vote - yes or no - deriving a column from existing col

cust_voting_df=cust_sel_DF.selectExpr("*","case when age>=18 then 'yes' else 'no' end as is_vote")
cust_voting_df.show(1)

In [0]:
# filter 


cust_non_voter_df=cust_voting_df.filter("is_vote='no'")
cust_non_voter_df.show()

cust_non_voter_df=cust_voting_df.where("profession='Pilot'")
cust_non_voter_df.show()


In [0]:
cust_final_df=cust_voting_df.withColumn("dt",date_format(current_date(),"yyyyMMdd"))

In [0]:
cust_final_df.show()

In [0]:
cust_final_df=cust_voting_df.withColumn("dt",date_format(current_date(),"yyyyMMdd"))
cust_final_df.show(1)
cust_final_df1=cust_voting_df.withColumn("dt",date_format(current_date(),"MM/yyyy/dd"))
cust_final_df1.show()

In [0]:
Group and Aggregations
groupBy - grouping column , dimension data, city , state , country
agg col - mesures , mostly numbers , sales ,salary


In [0]:
from pyspark.sql.functions import col,when,expr,current_date,date_format,count , min , max 
my_schema="custid int, fname string, lname string, age int, profession string"
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/custs_header",header=False,schema=my_schema) 
cust_df.show()
cust_df.printSchema()

In [0]:
# profession wise count 


# group - profession
# agg -> count , min , max 

In [0]:
from pyspark.sql.functions import col, min
group_prof_df=cust_df.groupBy("profession").count()

group_prof_df.show(5)
group_prof_df.printSchema()

# prof wise max age 
cust_df.show(1)
cust_df.printSchema()
group_prof_df = cust_df.groupBy("profession").min("age")

cust_df.count()
group_prof_df=cust_df.groupBy("profession").min("age")

group_prof_df.show()

In [0]:
from pyspark.sql.functions import col, min

group_prof_df1 = (
    cust_df.withColumn("age_int", col("age").cast("int"))
           
)
group_prof_df1.show()
cust_df.count()
group_prof_df1.count()
group_prof_df1.printSchema()

gdf=group_prof_df1.groupBy("profession"). min("age_int")
gdf.show()

In [0]:
cust_df.filter("profession is NULL").show()

In [0]:

# prof wise totnumber of records , min age , max age 

agg_df2=cust_df.groupBy("profession").agg(count("*").alias("tot_rec"),min("age"),max("age"))

agg_df2.show()

In [0]:
parse the customer data ad new column age_cat

--> young -- age till 25

--> mid age - 25 to 55

--> sr citizen -- above 55

tot_rec, min_age, max_age

In [0]:
from pyspark.sql.functions import col,when,expr,current_date,date_format,count , min , max 
 
my_schema="custid int, fname string, lname string, age int, profession string"
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/custs_header",header=False,schema=my_schema) 
cust_df.show(4)
cust_age_df=cust_df.withColumn("age_cat",expr("case when age <=25 then 'young' when age <=55 then 'mid age' else 'sr citizen' end " ))

cust_age_df.show(5)


In [0]:
cust_agg_df=cust_age_df.groupBy("age_cat").agg(count("*").alias("tot_rec"),min("age").alias("min_age"),max("age").alias("max_age"))

cust_agg_df.show(5) 

In [0]:
#ordering

cust_agg_df.orderBy("tot_rec",ascending=True).show()

In [0]:
# show - only showing the recors / print record , its return nothing -----------None - action 

# limit - returning a DF with  limited records  from source df  ---------------->datafrme  ---- transformation 

# take - return the limited record in python object list ----------------> list-----Action 


# show only 10 records 


cust_df.show(10)

# take 10 rec from df 
df=cust_df.limit(10)

df.show()

# take 


df1=cust_df.take(10)

print(df1)
display(df1)

In [0]:
df.take(10)

In [0]:
DATA Wrangling
-- join

In [0]:
Join
**1. left

right

inner**

full

self

cross

semi

anti

Spark internals - when shuffle happens this magic happen in spark internally , (performance side)

Broadcast join

Shuffle join

sort merge join

AQE - adoptive Query execution

join vs union
join - horizontal - adding columns - enriching with columns fron the other table - widenining

union - vertical - adding rows - making a tbale taller

left , left outer , right , right outer , full , full outer

In [0]:
# left  / left outer 
# all the records from the left and matching records from the right 
emp_data=[(100,"a1",25),(101,"a2",16),(102,"a3",35),(103,"a4",45),(104,"a5",25),(105,"a6",22)]

city_data=[(100,"chennai"),(105,"hyd"),(113,"delhi"),(125,"chennai")]

emp_df=spark.createDataFrame(emp_data,['id','name','age'])
city_df=spark.createDataFrame(city_data,['id','city'])

emp_df.show()
city_df.show()

In [0]:
# left join

# syntax :
# df1.join(df2,on,how)

# Df1.join(df2,condition,join_type)
# how -> join_type ----> default - inner 
# df.join(df1,on,how)
# df1.join(df2,on,how).join(df3,con,type)..

# join column name is same on both df , simily we can give column name in the condition 

# SELECT * FROM customers JOIN orders USING (customer_id);

emp_df.show()


In [0]:
city_df.show()
emp_city_df=emp_df.join(city_df,["id"],"left")
print("left Join result : all emp + matching city")
emp_city_df.show()

In [0]:
# case 2 - write using condiition 
# select * from emp left join city on emp.id=city.id
# id, name,age,id,city 

emp_city_df=emp_df.join(city_df,on=emp_df.id==city_df.id,how="left") 
emp_city_df.show()

In [0]:
from pyspark.sql.functions import col,expr
# case -3  
# join columns have different name 

# emp : id,name,age
# city : eid,city 
# join condition -> id=eid

# col() -> existing column 
# df.col_name -> columnn
# lit() -> literal value / hard coded 
city_df.show()

In [0]:
city_df_2 = city_df.withColumnRenamed("id","eid")

In [0]:
city_df_2.show()

In [0]:
emp_df.show()
-+----+---+
| id|name|age|
+---+----+---+
|100|  a1| 25|
|101|  a2| 16|
|102|  a3| 35|
|103|  a4| 45|
|104|  a5| 25|
|105|  a6| 22

city_df_2.show()

+---+-------+
|eid|   city|
+---+-------+
|100|chennai|
|105|    hyd|
|113|  delhi|
|125|chennai|
+---+-------+
emp_city_df=emp_df.join(city_df_2,on=emp_df.id==city_df_2.eid,how="left") 


emp_city_df=emp_df.join(city_df_2,on=col("id")==col("eid"),how="left") 
emp_city_df.show()


emp_city_df=emp_df.join(city_df_2,on=expr("id=eid"),how="left") 
emp_city_df.show()

In [0]:
# case 4 :
# if column names same in both dataframe - ambiqiuos 

# table alias 


emp_city_df=emp_df.alias("e").join(city_df_2.alias("c"),on=expr("e.id=c.eid"),how="left") 
emp_city_df.select("e.*","c.city").show()

In [0]:
# right / right outer
# all the records from the right and matching records from the left


emp_city_df=emp_df.join(city_df,["id"],"right")

emp_city_df.show()

In [0]:
# inner 

emp_city_df=emp_df.join(city_df,["id"],"inner")

emp_city_df.show()

In [0]:
### what happens if we have NULL  in join key 
### what happens if we have duplicate in join key 
# duplicate 

# cross join  

emp_data=[(100,"a1",25),(101,"a2",16),(102,"a3",35),(103,"a4",45),(104,"a5",25),(105,"a6",22),(105,"jeeva",25)]

city_data=[(100,"chennai"),(105,"hyd"),(113,"delhi"),(125,"chennai"),(100,"blr")]

emp_df=spark.createDataFrame(emp_data,['id','name','age'])
city_df=spark.createDataFrame(city_data,['id','city'])

emp_df.show()

city_df.show()

In [0]:
# inner
inner_join_df=emp_df.join(city_df,on="id",how="inner")
inner_join_df.show()

In [0]:
#left
left_join_df=emp_df.join(city_df,emp_df.id==city_df.id,"left")
left_join_df.show()
print(emp_df.count() ,left_join_df.count())

In [0]:
# cross join -> m record from table 1 , n records from table 2
# retrun m*n records

emp_data=[(100,"a1",25),(101,"a2",16)]
city_data=[(100,"chennai"),(105,"hyd")]

emp_df=spark.createDataFrame(emp_data,['id','name','age'])
city_df=spark.createDataFrame(city_data,['id','city'])

emp_df.show()

city_df.show()


cross_join_df=emp_df.crossJoin(city_df)

cross_join_df.show()

In [0]:
left semi - similar to exists
based on the matching condition , it will return only left side table

customer : cid ,cname
trans : tid,cid,sal_amt,sal_dt..
need a customer list , who made purchases last 2 months - left semi
need a customer id with total_trans, tans_amount ,from last 2 months data - inner
left anti - not exists
need a customer list , who didn't made any purchases last 2 months - left anti

In [0]:
emp_data=[(100,"a1",25),(101,"a2",16),(102,"a3",35),(103,"a4",45),(104,"a5",25),(105,"a6",22)]

city_data=[(100,"chennai"),(105,"hyd"),(113,"delhi"),(125,"chennai"),(105,"hyd2"),]

emp_df=spark.createDataFrame(emp_data,['id','name','age'])
city_df=spark.createDataFrame(city_data,['id','city'])

emp_df.show()

city_df.show()

In [0]:
left_join_df=emp_df.join(city_df,emp_df.id==city_df.id,'left')
display(left_join_df)

In [0]:
# only matching record from left side 
# employee who have city
left_semi_join_df=emp_df.join(city_df,emp_df.id==city_df.id,'left_semi')
display(left_semi_join_df)

In [0]:
# non matching record from left side   -- not exists 
# employees who dont have city
emp_df.show()
left_anti_join_df=emp_df.join(city_df,emp_df.id==city_df.id,'left_anti')
display(left_anti_join_df)

In [0]:
Windowing
window - set of related rows / grouped rows
similar to group by , but difference is group by collapse the record
diffrent types of windowing functions

Ranking - row_number,rank,dense_rank ...

Analytical - lead , lag , first , last ....

Aggregate - sum , avg , min ...

In [0]:
data =[ ('Lisa', 'Sales', 10000, 35),
          ('Evan', 'Sales', 32000, 38),
          ('Fred', 'Engineering', 21000, 28),
          ('Alex', 'Sales', 30000, 33),
          ('Tom', 'Engineering', 23000, 33),
          ('Jane', 'Marketing', 29000, 28),
          ('Jeff', 'Marketing', 35000, 38),
          ('Paul', 'Engineering', 29000, 23),
          ('Chloe', 'Engineering', 23000, 25)]

df = spark.createDataFrame(data, ['name', 'dept', 'salary', 'age'])
df.show()

In [0]:
we dont have any id/key column in the above data
source sending some key(natural key) column which is not unique on our table , i need to generate one key (surragate key(sk))
sequence number i need to generate in spark

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

df2=df.withColumn("emp_sk", monotonically_increasing_id())
df2.show()

# if data is in same partition , we will get unique sequence numbers 
# if the data in multiple partitions , we will get unique but numbers are not sequence 


In [0]:
window + ranking functions
row_number , rank , dense_rank

In [0]:
# syntax : function() over(window)
# window = partiton + ordering 
# row_number() over(partition by dept order by sal)


from pyspark.sql.functions import row_number,rank,dense_rank
from pyspark.sql.window import Window

df2.show(2)

# window 
#rank - it will skip the number if it ties
#dense rank - it will not skip the number if it ties
w_spaec = Window.partitionBy("dept").orderBy("salary")
df3=df2.withColumn("row_num",row_number().over(w_spaec)).withColumn("rank",rank().over(w_spaec)).withColumn("dense_rank",dense_rank().over(w_spaec))
df3.show()



 

In [0]:
# secon minimum salary in each dept
df4=df3.filter("dense_rank=2")
df4.show()

In [0]:
# sqL : -> row_number() over(partition by dept order by sal)

df2.select("*",row_number().over(Window.partitionBy("dept").orderBy("salary")).alias("row_num")).show()


In [0]:
data =[ ('Lisa', 'Sales', 10000, 35),
          ('Evan', 'Sales', 32000, 38),
          ('Fred', 'Engineering', 21000, 28),
          ('Alex', 'Sales', 30000, 33),
          ('Tom', 'Engineering', 23000, 33),
          ('Jane', 'Marketing', 29000, 28),
          ('Jeff', 'Marketing', 35000, 38),
          ('Paul', 'Engineering', 29000, 23),
          ('Chloe', 'Engineering', 23000, 25)]

df = spark.createDataFrame(data, ['name', 'dept', 'salary', 'age'])
df.show()

In [0]:
# global rank 

# row_number -> seq number / rank 

from pyspark.sql.functions import *
from pyspark.sql.window import Window

df.show()
from pyspark.sql.functions import desc
df2=df.select("*",row_number().over(window.orderBy(desc("age"))).alias("rnl"))
 

df2.show()
w_spec =Window.orderBy(asc("salary"))

df.select("*",row_number().over(w_spec).alias("rnk")).show()


In [0]:
# rank based on sal desc and if sal ties consider age also , least age give priority 
w_spaec_max=Window.partitionBy("dept").orderBy(desc("salary"),"age")
df_2=df.withColumn("rnk",dense_rank().over(w_spaec_max))
df_2.show()

In [0]:
df.write.saveAsTable("izwd37dev.wd37db.tbl_sal")

In [0]:
Top N analysys using standard SQL

In [0]:
%sql
-- global ranking 

select *,row_number() over(order by age desc) as rnk from izwd37dev.wd37db.tbl_sal;

-- global ranking based on age and sal

select *,row_number() over(order by age desc,salary) as rnk from izwd37dev.wd37db.tbl_sal

In [0]:
%sql
select *,dense_rank() over(partition by dept order by salary desc) as rnk from izwd37dev.wd37db.tbl_sal

In [0]:
Rank vs dense_rank vs row_number - interview note
second / third / nth - max/min sal - interview note

window - Analytical functions
lag
lead
first value
last_value

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/custs_header",header=False,inferSchema=True).toDF("custid","fname","lname","age","profession")
cust_df.show(2)

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/txns_2025.txt

In [0]:
strt1="txnid long,txndt string,custid long,amt float,cat string,prod string,city string,state string,spendby string"
txns_df=spark.read.schema(strt1).csv("/Volumes/izwd37dev/wd37db/rawdatta/txns_2025.txt",header=False)


In [0]:
txns_df.show(2)


# EDA - duplicate / nullability check

txns_df.count() # 95904

txns_df.distinct().count() # 95904


txns_df.describe().show() # 95904

In [0]:
txns_df.printSchema()

In [0]:
txns_final_df=txns_df.withColumn("txndt",to_date(col("txndt"),"MM-dd-yyyy"))
txns_final_df.show(5)

In [0]:
txns_final_df.printSchema()

In [0]:
# join 

joined_df=txns_final_df.join(cust_df,txns_final_df.custid==cust_df.custid,"inner")

joined_df.count()

txns_cust_df=joined_df.select(cust_df.custid,"txnid","fname","txndt","amt")

txns_cust_df.filter("custid=4000001").orderBy("txndt").show()

display(txns_cust_df.filter("custid=4000001").orderBy("txndt"))

In [0]:
str_col="custid"

w_spec=Window.partitionBy(str_col).orderBy("txndt")

txns_cust_df.withColumn("rnk",row_number().over(w_spec)).show()

# lead and lag 

# lead(col,offset,default=None)

# lag(col,offset,default=None)

In [0]:
txns_cust_df.show(2)

lag - Previous order value
lead - Next order value

In [0]:
txns_cust_df.withColumn("prvs_purchase",lag("amt").over(w_spec)).withColumn("nxt_purchase",lead("amt").over(w_spec)).withColumn("first_purchase",first("amt").over(w_spec)).show()

In [0]:
txns_cust_df.withColumn("prvs_purchase",lag("amt",2,0).over(w_spec)).withColumn("nxt_purchase",lead("amt",2,0).over(w_spec)).show()

In [0]:
#Grouping and Aggregate

data =[ ('Lisa', 'Sales', 10000, 35),
          ('Evan', 'Sales', 32000, 38),
          ('Fred', 'Engineering', 21000, 28),
          ('Alex', 'Sales', 30000, 33),
          ('Tom', 'Engineering', 23000, 33),
          ('Jane', 'Marketing', 29000, 28),
          ('Jeff', 'Marketing', 35000, 38),
          ('Paul', 'Engineering', 29000, 23),
          ('Chloe', 'Engineering', 23000, 25)]

df = spark.createDataFrame(data, ['name', 'dept', 'salary', 'age'])
df.show()

In [0]:
# dept wise total salary
from pyspark.sql.functions import sum, min, col, avg, max, min
df.groupBy("dept").agg(sum("salary").alias("total_sum"),
                       avg("salary").alias("avg_sal"),
                       min("salary").alias("min_sal"),
                       max("salary").alias("max_sal")
                       ).show()
                
                    

In [0]:
# identify salary of person based on dept avg salary higher , lower 
from pyspark.sql import Window
from pyspark.sql.functions import min, col
w_spec=Window.partitionBy("dept")
df.withColumn("min_salary",min("salary").over(w_spec)) \
    .withColumn("max_salary",max("salary").over(w_spec)) \
        .withColumn("sum_salary",sum("salary").over(w_spec)) \
            .withColumn("avg_salary",avg("salary").over(w_spec)).show()
#or


df.withColumn("min_salary",min("salary").over(Window.partitionBy("dept"))).show()

In [0]:
txns_cust_df.show(5)


txns_cust_df.groupBy(year("txndt"),month("txndt")).count().orderBy(desc("month(txndt)")).show()

In [0]:
df.show()


df.groupBy("dept","age").agg(sum("salary")).show()



In [0]:
df.rollup("dept","age").agg(sum("salary")).orderBy("dept").show()

In [0]:
set Opertations
union , intersection , subtract
pyspark DSL -> union = union all -> combining DF (similar)

In [0]:
data1=[(100,'raja',23),(101,'krish',35),(102,'jeff',30)]
data2=[(100,'raja',23),(501,'dino',35),(510,'shi',50)]

df1=spark.createDataFrame(data1,["id","name","age"])
df2=spark.createDataFrame(data2,["id","name","age"])

df1.show()
df2.show()

In [0]:
# union or unionByName or union All

df1.union(df2).show()

In [0]:
df1.unionByName(df2).show()

In [0]:
df1.unionAll(df2).show()

In [0]:
# intersection 
#intersect performs a set operation: it finds rows that appear in both DataFrames.

#Duplicate rows are removed (like SQL INTERSECT), so the result contains distinct common rows.

#Both DataFrames must have the same schema (same columns and types), otherwise Spark will throw an error.


df1.show()
df2.show()
df1.intersect(df2).show()

In [0]:
# subtract 
#returns the rows that are present in df1 but not in df2. It’s essentially the set difference operation.
df1.show()
df2.show()
df1.subtract(df2).show()